# Notebook 17 — Tableau Data Preparation

## Leadership and Management Book Recommendation System

### Objective

This notebook prepares the validated project outputs for visualization in Tableau.

All machine-learning development and model evaluation were completed in the preceding notebooks.

No models are trained, tuned, or modified in this notebook.

Instead, the objective is to create a small collection of clean, visualization-ready datasets that communicate the major findings of the project.

---

## Dashboard Objectives

The Tableau dashboard should allow users to understand:

1. the size and composition of the leadership and management book catalog;
2. metadata availability and data-source composition;
3. the thematic structure identified through topic clustering;
4. recommendation-system performance;
5. recommendation diversity and source exposure;
6. dimensionality-reduction and neural-network evaluation;
7. the final architecture selected for the recommendation application.

---

## Proposed Dashboard Structure

The Tableau output will support four analytical views.

### Dashboard 1 — Catalog Overview

Focus:

- total books;
- source composition;
- metadata coverage;
- publication-year distribution;
- authors and subjects;
- reader-engagement information where available.

### Dashboard 2 — Leadership Topic Landscape

Focus:

- 29 topic clusters;
- cluster sizes;
- cluster cohesion;
- representative books;
- major leadership and management themes.

### Dashboard 3 — Recommendation System Performance

Focus:

- catalog coverage;
- similarity by recommendation rank;
- topic diversity;
- source exposure;
- natural-language retrieval diagnostics;
- retrieval robustness.

### Dashboard 4 — Model Evaluation & Final Architecture

Focus:

- TF-IDF recommendation performance;
- SVD / LSA evaluation;
- autoencoder reconstruction and semantic preservation;
- model roles;
- final production architecture.

---

## Tableau Preparation Principle

The Tableau datasets should contain only fields required for visualization.

Large sparse TF-IDF matrices, neural-network tensors, model weights, serialized estimators, and raw NLP text representations are not required in Tableau.

The dashboard is an analytical communication layer rather than a model-execution environment.

All exported values must remain traceable to the validated outputs generated in the previous notebooks.

## 1. Project Setup and Artifact Discovery

The final processed datasets and evaluation outputs are inspected before Tableau-specific datasets are created.

This ensures that the visualization layer uses the validated project artifacts directly and avoids unnecessary duplication or manual re-entry of analytical results.

In [1]:
# ============================================================
# IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

FINAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "final"
)

TABLEAU_DIR = (
    PROJECT_ROOT
    / "data"
    / "tableau"
)


# Create Tableau directory if needed
TABLEAU_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("TABLEAU DATA PREPARATION — PROJECT SETUP")
print("=" * 80)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Processed directory exists:",
    PROCESSED_DIR.exists()
)

print(
    "Final directory exists:",
    FINAL_DIR.exists()
)

print(
    "Tableau directory:",
    TABLEAU_DIR
)

print(
    "Tableau directory exists:",
    TABLEAU_DIR.exists()
)

TABLEAU DATA PREPARATION — PROJECT SETUP
Project root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Processed directory exists: True
Final directory exists: True
Tableau directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/tableau
Tableau directory exists: True


In [2]:
# ============================================================
# DISCOVER TABLEAU-RELEVANT PROJECT ARTIFACTS
# ============================================================

tableau_keywords = [
    "books",
    "cluster",
    "recommendation",
    "evaluation",
    "exposure",
    "diversity",
    "similarity",
    "svd",
    "autoencoder",
    "llm",
    "coverage"
]


tableau_artifacts = sorted([
    path
    for path in PROCESSED_DIR.glob("*.csv")
    if any(
        keyword in path.name.lower()
        for keyword in tableau_keywords
    )
])


print("TABLEAU-RELEVANT PROCESSED ARTIFACTS")
print("=" * 90)

print(
    "Artifacts found:",
    len(tableau_artifacts)
)

print()

for path in tableau_artifacts:
    print(
        path.relative_to(PROJECT_ROOT)
    )


print("\nFINAL DATA ARTIFACTS")
print("=" * 90)

final_artifacts = sorted(
    FINAL_DIR.glob("*.csv")
)

print(
    "Artifacts found:",
    len(final_artifacts)
)

print()

for path in final_artifacts:
    print(
        path.relative_to(PROJECT_ROOT)
    )

TABLEAU-RELEVANT PROCESSED ARTIFACTS
Artifacts found: 23

data/processed/autoencoder_data_split.csv
data/processed/autoencoder_evaluation_summary.csv
data/processed/autoencoder_latent_32.csv
data/processed/autoencoder_training_history.csv
data/processed/books_features.csv
data/processed/books_nlp_features.csv
data/processed/books_with_final_topic_clusters.csv
data/processed/core_vs_enriched_coverage.csv
data/processed/core_vs_enriched_recommendations.csv
data/processed/final_cluster_representative_books.csv
data/processed/final_model_evaluation_summary.csv
data/processed/final_topic_cluster_summary.csv
data/processed/leadershipnow_books_clean.csv
data/processed/llm_grounding_catalog.csv
data/processed/llm_retrieval_diagnostic.csv
data/processed/llm_retrieval_robustness.csv
data/processed/recommendation_similarity_by_rank.csv
data/processed/recommendation_similarity_evaluation.csv
data/processed/recommendation_source_exposure.csv
data/processed/recommendation_topic_diversity.csv
data/pr

## 2. Dashboard 1 — Catalog Overview

The first Tableau dataset provides a book-level view of the final integrated catalog.

Its purpose is to describe the scope, composition, and metadata characteristics of the 2,067-book dataset used by the recommendation system.

The dataset should support analysis of:

- total catalog size;
- source composition;
- authorship;
- publication years;
- subjects and categories;
- reader-engagement measures;
- ratings where available;
- edition information;
- metadata depth and availability;
- topic-cluster membership where available.

The Tableau dataset is created from validated project outputs rather than the original raw API or scraped records.

No missing metadata is inferred or fabricated.

In [4]:
# ============================================================
# DASHBOARD 1 — INSPECT CATALOG INPUTS
# ============================================================

catalog_inputs = {
    "Books Master":
        FINAL_DIR / "books_master.csv",

    "Book Features":
        PROCESSED_DIR / "books_features.csv",

    "Topic Clusters":
        PROCESSED_DIR / "books_with_final_topic_clusters.csv"
}


catalog_data = {}


for name, path in catalog_inputs.items():

    df = pd.read_csv(path)

    catalog_data[name] = df

    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)

    print("Shape:", df.shape)

    print("\nColumns:")
    for column in df.columns:
        print(" -", column)

    print("\nBook ID status:")

    if "book_id" in df.columns:
        print("Unique book IDs:", df["book_id"].nunique())
        print("Missing book IDs:", df["book_id"].isna().sum())
        print("Duplicate book IDs:", df["book_id"].duplicated().sum())

    else:
        print("No book_id column found.")


BOOKS MASTER
Shape: (2067, 18)

Columns:
 - book_id
 - canonical_title
 - authors
 - description
 - subjects
 - first_publish_year
 - average_rating
 - ratings_count
 - edition_count
 - want_to_read_count
 - currently_reading_count
 - already_read_count
 - cover_url
 - openlibrary_key
 - source_openlibrary
 - source_leadershipnow
 - publication_year_observed
 - title_normalized

Book ID status:
Unique book IDs: 2067
Missing book IDs: 0
Duplicate book IDs: 0

BOOK FEATURES
Shape: (2067, 69)

Columns:
 - book_id
 - canonical_title
 - authors
 - description
 - subjects
 - first_publish_year
 - average_rating
 - ratings_count
 - edition_count
 - want_to_read_count
 - currently_reading_count
 - already_read_count
 - cover_url
 - openlibrary_key
 - source_openlibrary
 - source_leadershipnow
 - publication_year_observed
 - title_normalized
 - has_authors
 - has_description
 - has_subjects
 - has_cover
 - has_rating
 - has_engagement
 - has_extended_text
 - has_first_publish_year
 - has_obser

## 3. Create the Catalog Overview Dataset

The catalog overview uses `books_features.csv` as the primary book-level dataset because it contains all 2,067 canonical books together with the validated metadata and engineered analytical fields.

Topic-cluster information is added through a LEFT JOIN using `book_id`.

This preserves books that were not eligible for topic clustering.

For visualization purposes, books without a topic-cluster assignment are explicitly labeled **"Not Clustered"**. This is a presentation label only and does not represent an additional machine-learning cluster.

Only fields useful for Tableau analysis are retained. Large NLP text representations and model-specific feature columns are excluded.

In [5]:
# ============================================================
# CREATE TABLEAU CATALOG OVERVIEW
# ============================================================

books_features = catalog_data["Book Features"].copy()

topic_lookup = (
    catalog_data["Topic Clusters"][
        [
            "book_id",
            "topic_cluster",
            "cluster_label"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# LEFT JOIN topic information
# ------------------------------------------------------------

tableau_catalog = books_features.merge(
    topic_lookup,
    on="book_id",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# Presentation fields
# ------------------------------------------------------------

tableau_catalog["cluster_status"] = np.where(
    tableau_catalog["topic_cluster"].notna(),
    "Clustered",
    "Not Clustered"
)

tableau_catalog["cluster_label_display"] = (
    tableau_catalog["cluster_label"]
    .fillna("Not Clustered")
)


# Use nullable integer type so missing years remain missing
for column in [
    "first_publish_year",
    "publication_year_observed"
]:
    tableau_catalog[column] = pd.to_numeric(
        tableau_catalog[column],
        errors="coerce"
    ).astype("Int64")


# ------------------------------------------------------------
# Select Tableau-relevant fields
# ------------------------------------------------------------

catalog_columns = [
    # Identification
    "book_id",
    "canonical_title",
    "authors",

    # Source
    "source_group",
    "source_openlibrary",
    "source_leadershipnow",

    # Publication
    "first_publish_year",
    "publication_year_observed",
    "book_age_from_first_publish",

    # Reader metrics
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "total_reader_engagement",

    # Edition metadata
    "format",
    "page_count",

    # Metadata availability
    "has_authors",
    "has_description",
    "has_subjects",
    "has_cover",
    "has_rating",
    "has_engagement",
    "has_first_publish_year",
    "has_observed_publication_year",
    "has_edition_metadata",
    "has_page_count",

    # Content depth
    "content_component_count",
    "content_depth",
    "core_word_count",
    "enriched_word_count",
    "additional_semantic_words",

    # Topic clustering
    "topic_cluster",
    "cluster_label",
    "cluster_status",
    "cluster_label_display",

    # Display
    "cover_url"
]


tableau_catalog = tableau_catalog[
    catalog_columns
].copy()


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("TABLEAU CATALOG OVERVIEW")
print("=" * 90)

print("Rows:", len(tableau_catalog))
print("Columns:", tableau_catalog.shape[1])

print(
    "Unique books:",
    tableau_catalog["book_id"].nunique()
)

print(
    "Duplicate book IDs:",
    tableau_catalog["book_id"].duplicated().sum()
)

print(
    "Clustered books:",
    (tableau_catalog["cluster_status"] == "Clustered").sum()
)

print(
    "Not clustered:",
    (tableau_catalog["cluster_status"] == "Not Clustered").sum()
)

print("\nSOURCE COMPOSITION")
print("-" * 50)

display(
    tableau_catalog[
        "source_group"
    ]
    .value_counts(dropna=False)
    .rename_axis("source_group")
    .reset_index(name="books")
)

print("\nCONTENT DEPTH")
print("-" * 50)

display(
    tableau_catalog[
        "content_depth"
    ]
    .value_counts(dropna=False)
    .rename_axis("content_depth")
    .reset_index(name="books")
)

print("\nPreview:")
display(
    tableau_catalog.head()
)

TABLEAU CATALOG OVERVIEW
Rows: 2067
Columns: 38
Unique books: 2067
Duplicate book IDs: 0
Clustered books: 1884
Not clustered: 183

SOURCE COMPOSITION
--------------------------------------------------


,source_group,books
0,LeadershipNow only,1117
1,Open Library only,947
2,Both,3



CONTENT DEPTH
--------------------------------------------------


,content_depth,books
0,Basic,1240
1,Enriched,635
2,Rich,189
3,Minimal,3



Preview:


,book_id,canonical_title,authors,source_group,source_openlibrary,source_leadershipnow,first_publish_year,publication_year_observed,book_age_from_first_publish,average_rating,...,content_component_count,content_depth,core_word_count,enriched_word_count,additional_semantic_words,topic_cluster,cluster_label,cluster_status,cluster_label_display,cover_url
0,BOOK00001,Principle-Centered Leadership,['Stephen R. Covey'],Open Library only,True,False,1989,<NA>,37.0,4.500000,...,4,Rich,5,199,194,7.0,Broad Leadership and Success,Clustered,Broad Leadership and Success,https://covers.openlibrary.org/b/id/10858615-L...
1,BOOK00002,Leadership in Organizations,['Gary A. Yukl'],Open Library only,True,False,1981,<NA>,45.0,5.000000,...,3,Enriched,6,51,45,4.0,Decision Making,Clustered,Decision Making,https://covers.openlibrary.org/b/id/87719-L.jpg
2,BOOK00003,Kepemimpinan =,['Karjadi M.'],Open Library only,True,False,1977,<NA>,49.0,1.000000,...,3,Enriched,4,5,1,25.0,Transformational Leadership,Clustered,Transformational Leadership,https://covers.openlibrary.org/b/id/14420782-L...
3,BOOK00004,Spiritual leadership,['J. Oswald Sanders'],Open Library only,True,False,1967,<NA>,59.0,5.000000,...,3,Enriched,5,10,5,25.0,Transformational Leadership,Clustered,Transformational Leadership,https://covers.openlibrary.org/b/id/570509-L.jpg
4,BOOK00005,Leadership,['Peter Guy Northouse'],Open Library only,True,False,1997,<NA>,29.0,3.666667,...,3,Enriched,4,18,14,25.0,Transformational Leadership,Clustered,Transformational Leadership,https://covers.openlibrary.org/b/id/3859675-L.jpg


## 4. Catalog KPI and Metadata Coverage

A second Tableau dataset summarizes the principal catalog-level indicators.

Unlike the book-level catalog dataset, this file contains aggregated measures designed for KPI cards and metadata-coverage visualizations.

Metadata coverage is calculated directly from the validated book-level availability indicators.

The resulting measures describe the availability of information in the catalog and should not be interpreted as missing-data imputation or model performance.

In [6]:
# ============================================================
# DASHBOARD 1 — CATALOG KPI AND METADATA COVERAGE
# ============================================================

total_books = len(tableau_catalog)


# ------------------------------------------------------------
# Metadata coverage definitions
# ------------------------------------------------------------

coverage_fields = {
    "Authors": "has_authors",
    "Description": "has_description",
    "Subjects": "has_subjects",
    "Cover Image": "has_cover",
    "Rating": "has_rating",
    "Reader Engagement": "has_engagement",
    "First Publish Year": "has_first_publish_year",
    "Observed Publication Year": "has_observed_publication_year",
    "Edition Metadata": "has_edition_metadata",
    "Page Count": "has_page_count"
}


coverage_rows = []

for label, column in coverage_fields.items():

    available = int(
        tableau_catalog[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    coverage_rows.append({
        "metadata_field": label,
        "available_books": available,
        "missing_books": total_books - available,
        "coverage_pct": available / total_books * 100
    })


tableau_metadata_coverage = pd.DataFrame(
    coverage_rows
)


# ------------------------------------------------------------
# Catalog KPI table
# ------------------------------------------------------------

catalog_kpis = pd.DataFrame([
    {
        "kpi": "Total Books",
        "value": total_books,
        "unit": "books"
    },
    {
        "kpi": "Clustered Books",
        "value": int(
            (tableau_catalog["cluster_status"] == "Clustered").sum()
        ),
        "unit": "books"
    },
    {
        "kpi": "Topic Clusters",
        "value": int(
            tableau_catalog["topic_cluster"].nunique()
        ),
        "unit": "clusters"
    },
    {
        "kpi": "Open Library Only",
        "value": int(
            (tableau_catalog["source_group"] == "Open Library only").sum()
        ),
        "unit": "books"
    },
    {
        "kpi": "LeadershipNow Only",
        "value": int(
            (tableau_catalog["source_group"] == "LeadershipNow only").sum()
        ),
        "unit": "books"
    },
    {
        "kpi": "Both Sources",
        "value": int(
            (tableau_catalog["source_group"] == "Both").sum()
        ),
        "unit": "books"
    }
])


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("CATALOG KPIs")
print("=" * 80)

display(catalog_kpis)


print("\nMETADATA COVERAGE")
print("=" * 80)

display(
    tableau_metadata_coverage
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

CATALOG KPIs


,kpi,value,unit
0,Total Books,2067,books
1,Clustered Books,1884,books
2,Topic Clusters,29,clusters
3,Open Library Only,947,books
4,LeadershipNow Only,1117,books
5,Both Sources,3,books



METADATA COVERAGE


,metadata_field,available_books,missing_books,coverage_pct
0,Authors,2059,8,99.612966
1,Cover Image,1888,179,91.340106
2,Edition Metadata,1120,947,54.184809
3,Page Count,1119,948,54.136430
4,Observed Publication Year,1117,950,54.039671
5,First Publish Year,947,1120,45.815191
6,Subjects,826,1241,39.961297
7,Reader Engagement,807,1260,39.042090
8,Rating,282,1785,13.642961
9,Description,192,1875,9.288824


## 5. Export Dashboard 1 — Catalog Overview

The validated catalog-level datasets are exported to the Tableau data directory.

Three files are produced:

1. `tableau_catalog_overview.csv` — one row per canonical book;
2. `tableau_catalog_kpis.csv` — high-level catalog indicators;
3. `tableau_metadata_coverage.csv` — metadata availability and coverage.

Before export, topic-cluster identifiers are converted to nullable integers so that Tableau treats them as discrete cluster identifiers rather than continuous decimal values.

In [8]:
# ============================================================
# EXPORT DASHBOARD 1 TABLEAU DATA
# ============================================================

# Clean cluster identifier for Tableau
tableau_catalog["topic_cluster"] = (
    pd.to_numeric(
        tableau_catalog["topic_cluster"],
        errors="coerce"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

CATALOG_PATH = (
    TABLEAU_DIR
    / "tableau_catalog_overview.csv"
)

KPI_PATH = (
    TABLEAU_DIR
    / "tableau_catalog_kpis.csv"
)

COVERAGE_PATH = (
    TABLEAU_DIR
    / "tableau_metadata_coverage.csv"
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

tableau_catalog.to_csv(
    CATALOG_PATH,
    index=False
)

catalog_kpis.to_csv(
    KPI_PATH,
    index=False
)

tableau_metadata_coverage.to_csv(
    COVERAGE_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload validation
# ------------------------------------------------------------

catalog_reload = pd.read_csv(CATALOG_PATH)
kpi_reload = pd.read_csv(KPI_PATH)
coverage_reload = pd.read_csv(COVERAGE_PATH)


print("DASHBOARD 1 TABLEAU EXPORT")
print("=" * 90)

print(
    "Catalog:",
    catalog_reload.shape,
    CATALOG_PATH.name
)

print(
    "KPIs:",
    kpi_reload.shape,
    KPI_PATH.name
)

print(
    "Metadata coverage:",
    coverage_reload.shape,
    COVERAGE_PATH.name
)

print("\nVALIDATION")
print("-" * 50)

print(
    "Catalog rows correct:",
    len(catalog_reload) == 2067
)

print(
    "Unique book IDs:",
    catalog_reload["book_id"].nunique() == 2067
)

print(
    "Duplicate book IDs:",
    catalog_reload["book_id"].duplicated().sum()
)

print(
    "Clustered books:",
    (
        catalog_reload["cluster_status"] == "Clustered"
    ).sum()
)

print(
    "Topic clusters:",
    catalog_reload["topic_cluster"].nunique()
)

print(
    "KPI rows:",
    len(kpi_reload)
)

print(
    "Coverage rows:",
    len(coverage_reload)
)

DASHBOARD 1 TABLEAU EXPORT
Catalog: (2067, 38) tableau_catalog_overview.csv
KPIs: (6, 3) tableau_catalog_kpis.csv
Metadata coverage: (10, 4) tableau_metadata_coverage.csv

VALIDATION
--------------------------------------------------
Catalog rows correct: True
Unique book IDs: True
Duplicate book IDs: 0
Clustered books: 1884
Topic clusters: 29
KPI rows: 6
Coverage rows: 10


## 6. Dashboard 2 — Leadership Topic Landscape

The second Tableau dashboard presents the thematic structure discovered through unsupervised topic clustering.

The final clustering solution contains 29 clusters representing 1,884 books.

This dashboard is intended to answer questions such as:

- What leadership and management themes are represented in the catalog?
- Which themes contain the largest number of books?
- Which clusters are relatively compact or broad?
- What terms characterize each topic?
- Which books are representative of each topic?
- How are the two primary data sources distributed across the topic landscape?

The clusters should be interpreted as data-driven thematic groupings rather than definitive classifications of leadership literature.

Cluster labels were assigned for interpretability after inspection of cluster terms and representative books.

In [9]:
# ============================================================
# DASHBOARD 2 — INSPECT TOPIC LANDSCAPE INPUTS
# ============================================================

cluster_summary = pd.read_csv(
    PROCESSED_DIR
    / "final_topic_cluster_summary.csv"
)

representative_books = pd.read_csv(
    PROCESSED_DIR
    / "final_cluster_representative_books.csv"
)

clustered_books = pd.read_csv(
    PROCESSED_DIR
    / "books_with_final_topic_clusters.csv"
)


# ------------------------------------------------------------
# Cluster summary
# ------------------------------------------------------------

print("CLUSTER SUMMARY")
print("=" * 100)

print("Shape:", cluster_summary.shape)

print("\nColumns:")
for column in cluster_summary.columns:
    print(" -", column)

print("\nComplete cluster list:")

display(
    cluster_summary[
        [
            "cluster",
            "cluster_label",
            "cluster_size",
            "mean_distance_to_centroid",
            "cohesion_group"
        ]
    ]
    .sort_values("cluster")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Representative books
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REPRESENTATIVE BOOKS")
print("=" * 100)

print("Shape:", representative_books.shape)

print("\nColumns:")
for column in representative_books.columns:
    print(" -", column)

print("\nPreview:")

display(
    representative_books.head(10)
)


# ------------------------------------------------------------
# Clustered books
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CLUSTERED BOOKS")
print("=" * 100)

print("Shape:", clustered_books.shape)

print(
    "Books:",
    clustered_books["book_id"].nunique()
)

print(
    "Clusters:",
    clustered_books["topic_cluster"].nunique()
)

print(
    "Missing cluster labels:",
    clustered_books["cluster_label"].isna().sum()
)

CLUSTER SUMMARY
Shape: (29, 10)

Columns:
 - cluster
 - cluster_label
 - cluster_size
 - top_terms
 - representative_titles
 - average_centroid_distance
 - mean_distance_to_centroid
 - median_distance_to_centroid
 - max_distance_to_centroid
 - cohesion_group

Complete cluster list:


,cluster,cluster_label,cluster_size,mean_distance_to_centroid,cohesion_group
0,0,Entrepreneurial and Growth Mindset,82,0.794508,Broadest
1,1,Organizational Behavior,56,0.396443,Moderately compact
2,2,Team Leadership,33,0.446914,Moderately compact
3,3,Future Thinking and Personal Development,70,0.775839,Broadest
4,4,Decision Making,33,0.510835,Moderately broad
5,5,Leader Identity and Practice,33,0.380652,Moderately compact
6,6,Communication,47,0.534248,Moderately broad
7,7,Broad Leadership and Success,185,0.877948,Broadest
8,8,Servant Leadership,42,0.399867,Moderately compact
9,9,Change Management,42,0.478248,Moderately broad



REPRESENTATIVE BOOKS
Shape: (145, 8)

Columns:
 - cluster
 - cluster_size
 - rank
 - book_id
 - title
 - authors
 - distance_to_centroid
 - cluster_label

Preview:


,cluster,cluster_size,rank,book_id,title,authors,distance_to_centroid,cluster_label
0,0,82,1,BOOK02062,A Platform Mindset,['Marcus Fontoura'],0.379655,Entrepreneurial and Growth Mindset
1,0,82,2,BOOK01663,Unstoppable Mindset,['Alden Mills'],0.379655,Entrepreneurial and Growth Mindset
2,0,82,3,BOOK01664,The Hacker Mindset,['Garrett Gee'],0.379655,Entrepreneurial and Growth Mindset
3,0,82,4,BOOK01631,Aliveness Mindset,['Jack Craven'],0.379655,Entrepreneurial and Growth Mindset
4,0,82,5,BOOK01670,The Entrepreneurial Mindset Advantage,['Gary G. Schoeniger'],0.524835,Entrepreneurial and Growth Mindset
5,1,56,1,BOOK00674,Organizational Behavior,"['Christopher P. Neck', 'Jeffery D. Houghton',...",0.091832,Organizational Behavior
6,1,56,2,BOOK00703,Organizational behavior,"['Afsaneh Nahavandi', 'Ali R. Malekzadeh']",0.091832,Organizational Behavior
7,1,56,3,BOOK00688,Organizational Behavior,['Ricky W. Griffin'],0.091832,Organizational Behavior
8,1,56,4,BOOK00679,Organizational behavior,"['Angelo Kinicki', 'Robert Kreitner']",0.091832,Organizational Behavior
9,1,56,5,BOOK00706,Organizational Behavior,"['Mary Uhl-Bien', 'Ronald F. Piccolo', 'Scherm...",0.098853,Organizational Behavior



CLUSTERED BOOKS
Shape: (1884, 73)
Books: 1884
Clusters: 29
Missing cluster labels: 0


## 7. Create the Leadership Topic Landscape Dataset

The topic-landscape dataset combines the final cluster summary with source-composition information calculated from the 1,884 clustered books.

Each row represents one of the 29 validated topic clusters.

The dataset includes:

- cluster identifier and label;
- number and percentage of clustered books;
- cluster cohesion measures;
- top terms;
- representative titles;
- source composition.

The original cluster assignments and labels are preserved without modification.

Cluster distance measures describe cohesion within the fitted representation. Lower distances indicate relatively more compact clusters, while higher distances indicate broader semantic groupings.

These measures should be interpreted comparatively within this clustering solution rather than as universal measures of topic quality.

In [10]:
# ============================================================
# CREATE TABLEAU TOPIC LANDSCAPE
# ============================================================

# ------------------------------------------------------------
# Source composition by cluster
# ------------------------------------------------------------

cluster_source_counts = (
    clustered_books
    .groupby(
        ["topic_cluster", "source_group"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)


# Ensure all expected source columns exist
for source in [
    "LeadershipNow only",
    "Open Library only",
    "Both"
]:
    if source not in cluster_source_counts.columns:
        cluster_source_counts[source] = 0


cluster_source_counts = (
    cluster_source_counts
    .rename(
        columns={
            "topic_cluster": "cluster",
            "LeadershipNow only": "leadershipnow_books",
            "Open Library only": "openlibrary_books",
            "Both": "both_source_books"
        }
    )
)


# ------------------------------------------------------------
# Merge with validated cluster summary
# ------------------------------------------------------------

tableau_topic_clusters = cluster_summary.merge(
    cluster_source_counts,
    on="cluster",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# Additional Tableau measures
# ------------------------------------------------------------

total_clustered_books = int(
    cluster_summary["cluster_size"].sum()
)

tableau_topic_clusters["cluster_share_pct"] = (
    tableau_topic_clusters["cluster_size"]
    / total_clustered_books
    * 100
)

tableau_topic_clusters["leadershipnow_share_pct"] = (
    tableau_topic_clusters["leadershipnow_books"]
    / tableau_topic_clusters["cluster_size"]
    * 100
)

tableau_topic_clusters["openlibrary_share_pct"] = (
    tableau_topic_clusters["openlibrary_books"]
    / tableau_topic_clusters["cluster_size"]
    * 100
)

tableau_topic_clusters["both_source_share_pct"] = (
    tableau_topic_clusters["both_source_books"]
    / tableau_topic_clusters["cluster_size"]
    * 100
)


# ------------------------------------------------------------
# Select visualization fields
# ------------------------------------------------------------

topic_columns = [
    "cluster",
    "cluster_label",
    "cluster_size",
    "cluster_share_pct",
    "top_terms",
    "representative_titles",
    "average_centroid_distance",
    "mean_distance_to_centroid",
    "median_distance_to_centroid",
    "max_distance_to_centroid",
    "cohesion_group",
    "leadershipnow_books",
    "openlibrary_books",
    "both_source_books",
    "leadershipnow_share_pct",
    "openlibrary_share_pct",
    "both_source_share_pct"
]


tableau_topic_clusters = (
    tableau_topic_clusters[
        topic_columns
    ]
    .sort_values(
        "cluster"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("TABLEAU TOPIC LANDSCAPE")
print("=" * 100)

print(
    "Clusters:",
    len(tableau_topic_clusters)
)

print(
    "Unique cluster IDs:",
    tableau_topic_clusters["cluster"].nunique()
)

print(
    "Total clustered books:",
    tableau_topic_clusters["cluster_size"].sum()
)

print(
    "Cluster share total:",
    tableau_topic_clusters["cluster_share_pct"].sum()
)

print(
    "Source-count total:",
    tableau_topic_clusters[
        [
            "leadershipnow_books",
            "openlibrary_books",
            "both_source_books"
        ]
    ].sum().sum()
)

print(
    "Missing cluster labels:",
    tableau_topic_clusters["cluster_label"].isna().sum()
)


print("\nTOP 10 LARGEST TOPIC CLUSTERS")
print("-" * 100)

display(
    tableau_topic_clusters[
        [
            "cluster",
            "cluster_label",
            "cluster_size",
            "cluster_share_pct",
            "cohesion_group",
            "mean_distance_to_centroid"
        ]
    ]
    .sort_values(
        "cluster_size",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

TABLEAU TOPIC LANDSCAPE
Clusters: 29
Unique cluster IDs: 29
Total clustered books: 1884
Cluster share total: 100.0
Source-count total: 1884
Missing cluster labels: 0

TOP 10 LARGEST TOPIC CLUSTERS
----------------------------------------------------------------------------------------------------


,cluster,cluster_label,cluster_size,cluster_share_pct,cohesion_group,mean_distance_to_centroid
0,28,CEO and Executive Transformation,285,15.127389,Broadest,0.896801
1,7,Broad Leadership and Success,185,9.819533,Broadest,0.877948
2,11,Organizational Culture and Trust,184,9.766454,Broadest,0.874449
3,25,Transformational Leadership,108,5.732484,Broadest,0.596423
4,27,Strategic Management and Planning,82,4.352442,Moderately broad,0.500045
5,0,Entrepreneurial and Growth Mindset,82,4.352442,Broadest,0.794508
6,13,General Business and Management,72,3.821656,Broadest,0.649138
7,3,Future Thinking and Personal Development,70,3.715499,Broadest,0.775839
8,20,Human Resource Management,57,3.025478,Most compact,0.342664
9,1,Organizational Behavior,56,2.972399,Moderately compact,0.396443


## 8. Representative Books by Topic Cluster

Each topic cluster contains five representative books selected according to proximity to the cluster centroid.

These books provide concrete examples of the titles most closely associated with each cluster's central semantic representation.

The representative-book table is kept separate from the cluster-level summary because it has a one-to-many relationship:

**1 Topic Cluster → 5 Representative Books**

This structure allows Tableau to use cluster selection as an interactive filter while displaying representative titles, authors, and centroid distances.

Representative books illustrate the fitted cluster structure and should not be interpreted as rankings of book quality.

In [11]:
# ============================================================
# CREATE TABLEAU REPRESENTATIVE BOOKS
# ============================================================

tableau_representative_books = (
    representative_books[
        [
            "cluster",
            "cluster_label",
            "cluster_size",
            "rank",
            "book_id",
            "title",
            "authors",
            "distance_to_centroid"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Add source and display metadata
# ------------------------------------------------------------

book_display_lookup = (
    tableau_catalog[
        [
            "book_id",
            "source_group",
            "first_publish_year",
            "publication_year_observed",
            "average_rating",
            "ratings_count",
            "cover_url"
        ]
    ]
    .copy()
)


tableau_representative_books = (
    tableau_representative_books
    .merge(
        book_display_lookup,
        on="book_id",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        ["cluster", "rank"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

representatives_per_cluster = (
    tableau_representative_books
    .groupby("cluster")
    .size()
)


print("TABLEAU REPRESENTATIVE BOOKS")
print("=" * 100)

print(
    "Rows:",
    len(tableau_representative_books)
)

print(
    "Clusters represented:",
    tableau_representative_books["cluster"].nunique()
)

print(
    "Unique books:",
    tableau_representative_books["book_id"].nunique()
)

print(
    "Minimum representatives per cluster:",
    representatives_per_cluster.min()
)

print(
    "Maximum representatives per cluster:",
    representatives_per_cluster.max()
)

print(
    "Missing titles:",
    tableau_representative_books["title"].isna().sum()
)

print(
    "Missing cluster labels:",
    tableau_representative_books["cluster_label"].isna().sum()
)

print(
    "Missing source groups:",
    tableau_representative_books["source_group"].isna().sum()
)


print("\nSAMPLE — FIRST THREE CLUSTERS")
print("-" * 100)

display(
    tableau_representative_books[
        tableau_representative_books[
            "cluster"
        ].isin([0, 1, 2])
    ]
)

TABLEAU REPRESENTATIVE BOOKS
Rows: 145
Clusters represented: 29
Unique books: 145
Minimum representatives per cluster: 5
Maximum representatives per cluster: 5
Missing titles: 0
Missing cluster labels: 0
Missing source groups: 0

SAMPLE — FIRST THREE CLUSTERS
----------------------------------------------------------------------------------------------------


,cluster,cluster_label,cluster_size,rank,book_id,title,authors,distance_to_centroid,source_group,first_publish_year,publication_year_observed,average_rating,ratings_count,cover_url
0,0,Entrepreneurial and Growth Mindset,82,1,BOOK02062,A Platform Mindset,['Marcus Fontoura'],0.379655,LeadershipNow only,<NA>,2025,NaN,NaN,https://www.leadershipnow.com/leadershop/image...
1,0,Entrepreneurial and Growth Mindset,82,2,BOOK01663,Unstoppable Mindset,['Alden Mills'],0.379655,LeadershipNow only,<NA>,2024,NaN,NaN,https://www.leadershipnow.com/leadershop/image...
2,0,Entrepreneurial and Growth Mindset,82,3,BOOK01664,The Hacker Mindset,['Garrett Gee'],0.379655,LeadershipNow only,<NA>,2024,NaN,NaN,https://www.leadershipnow.com/leadershop/image...
3,0,Entrepreneurial and Growth Mindset,82,4,BOOK01631,Aliveness Mindset,['Jack Craven'],0.379655,LeadershipNow only,<NA>,2024,NaN,NaN,https://www.leadershipnow.com/leadershop/image...
4,0,Entrepreneurial and Growth Mindset,82,5,BOOK01670,The Entrepreneurial Mindset Advantage,['Gary G. Schoeniger'],0.524835,LeadershipNow only,<NA>,2024,NaN,NaN,https://www.leadershipnow.com/leadershop/image...
5,1,Organizational Behavior,56,1,BOOK00674,Organizational Behavior,"['Christopher P. Neck', 'Jeffery D. Houghton',...",0.091832,Open Library only,2015,<NA>,NaN,NaN,https://covers.openlibrary.org/b/id/8904889-L.jpg
6,1,Organizational Behavior,56,2,BOOK00703,Organizational behavior,"['Afsaneh Nahavandi', 'Ali R. Malekzadeh']",0.091832,Open Library only,1998,<NA>,NaN,NaN,https://covers.openlibrary.org/b/id/90010-L.jpg
7,1,Organizational Behavior,56,3,BOOK00688,Organizational Behavior,['Ricky W. Griffin'],0.091832,Open Library only,1986,<NA>,NaN,NaN,https://covers.openlibrary.org/b/id/1325242-L.jpg
8,1,Organizational Behavior,56,4,BOOK00679,Organizational behavior,"['Angelo Kinicki', 'Robert Kreitner']",0.091832,Open Library only,2003,<NA>,NaN,NaN,https://covers.openlibrary.org/b/id/8271863-L.jpg
9,1,Organizational Behavior,56,5,BOOK00706,Organizational Behavior,"['Mary Uhl-Bien', 'Ronald F. Piccolo', 'Scherm...",0.098853,Open Library only,2013,<NA>,NaN,NaN,https://covers.openlibrary.org/b/id/13985922-L...


In [12]:
# ============================================================
# EXPORT DASHBOARD 2 TABLEAU DATA
# ============================================================

TOPIC_PATH = (
    TABLEAU_DIR
    / "tableau_topic_clusters.csv"
)

REPRESENTATIVE_PATH = (
    TABLEAU_DIR
    / "tableau_representative_books.csv"
)


tableau_topic_clusters.to_csv(
    TOPIC_PATH,
    index=False
)

tableau_representative_books.to_csv(
    REPRESENTATIVE_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload validation
# ------------------------------------------------------------

topic_reload = pd.read_csv(
    TOPIC_PATH
)

representative_reload = pd.read_csv(
    REPRESENTATIVE_PATH
)


dashboard2_checks = {
    "29 topic clusters":
        len(topic_reload) == 29,

    "29 unique cluster IDs":
        topic_reload["cluster"].nunique() == 29,

    "1884 clustered books represented":
        topic_reload["cluster_size"].sum() == 1884,

    "Cluster shares total 100%":
        np.isclose(
            topic_reload["cluster_share_pct"].sum(),
            100.0
        ),

    "145 representative records":
        len(representative_reload) == 145,

    "All 29 clusters have representatives":
        representative_reload["cluster"].nunique() == 29,

    "Exactly 5 representatives per cluster":
        (
            representative_reload
            .groupby("cluster")
            .size()
            .eq(5)
            .all()
        )
}


dashboard2_validation = pd.DataFrame(
    dashboard2_checks.items(),
    columns=[
        "validation_check",
        "passed"
    ]
)


print("DASHBOARD 2 TABLEAU EXPORT")
print("=" * 100)

print(
    "Topic clusters:",
    topic_reload.shape,
    TOPIC_PATH.name
)

print(
    "Representative books:",
    representative_reload.shape,
    REPRESENTATIVE_PATH.name
)

print("\nVALIDATION")
print("-" * 100)

display(
    dashboard2_validation
)

print(
    "\nAll Dashboard 2 checks passed:",
    bool(
        dashboard2_validation["passed"].all()
    )
)

DASHBOARD 2 TABLEAU EXPORT
Topic clusters: (29, 17) tableau_topic_clusters.csv
Representative books: (145, 14) tableau_representative_books.csv

VALIDATION
----------------------------------------------------------------------------------------------------


,validation_check,passed
0,29 topic clusters,True
1,29 unique cluster IDs,True
2,1884 clustered books represented,True
3,Cluster shares total 100%,True
4,145 representative records,True
5,All 29 clusters have representatives,True
6,Exactly 5 representatives per cluster,True



All Dashboard 2 checks passed: True


## 10. Dashboard 3 — Recommendation System Performance

The third Tableau dashboard evaluates the behavior of the final production recommendation system:

**Enriched TF-IDF + Cosine Similarity**

The purpose of this dashboard is not to report supervised prediction accuracy because the recommendation system has no labeled relevance target.

Instead, the dashboard evaluates the system using measures appropriate to content-based retrieval:

- catalog coverage;
- recommendation similarity by rank;
- topic diversity;
- source exposure;
- natural-language retrieval diagnostics;
- retrieval robustness.

The dashboard also highlights an important limitation identified during evaluation: recommendation exposure differs between the two primary data sources.

This pattern is reported as **source-associated recommendation behavior** because metadata availability and richness differ substantially between the sources.

The dashboard therefore communicates both recommendation performance and system limitations.

In [13]:
# ============================================================
# DASHBOARD 3 — LOAD AND INSPECT RECOMMENDATION INPUTS
# ============================================================

similarity_by_rank = pd.read_csv(
    PROCESSED_DIR
    / "recommendation_similarity_by_rank.csv"
)

recommendation_diversity = pd.read_csv(
    PROCESSED_DIR
    / "recommendation_topic_diversity.csv"
)

source_exposure = pd.read_csv(
    PROCESSED_DIR
    / "recommendation_source_exposure.csv"
)

retrieval_diagnostic = pd.read_csv(
    PROCESSED_DIR
    / "llm_retrieval_diagnostic.csv"
)

retrieval_robustness = pd.read_csv(
    PROCESSED_DIR
    / "llm_retrieval_robustness.csv"
)


recommendation_inputs = {
    "Similarity by Rank": similarity_by_rank,
    "Topic Diversity": recommendation_diversity,
    "Source Exposure": source_exposure,
    "Retrieval Diagnostic": retrieval_diagnostic,
    "Retrieval Robustness": retrieval_robustness
}


for name, df in recommendation_inputs.items():

    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)

    print("Shape:", df.shape)

    print("\nColumns:")
    for column in df.columns:
        print(" -", column)

    print("\nPreview:")
    display(df.head())


SIMILARITY BY RANK
Shape: (10, 4)

Columns:
 - recommendation_rank
 - count
 - mean
 - median

Preview:


,recommendation_rank,count,mean,median
0,1,2040,0.513143,0.494901
1,2,2026,0.410047,0.396190
2,3,1994,0.361421,0.348542
3,4,1976,0.327658,0.321637
4,5,1961,0.300902,0.292859



TOPIC DIVERSITY
Shape: (2040, 5)

Columns:
 - query_book_id
 - recommendations_returned
 - recommendations_with_cluster
 - unique_topic_clusters
 - cluster_diversity_ratio

Preview:


,query_book_id,recommendations_returned,recommendations_with_cluster,unique_topic_clusters,cluster_diversity_ratio
0,BOOK00001,10,10,7,0.700000
1,BOOK00002,10,10,4,0.400000
2,BOOK00003,10,10,5,0.500000
3,BOOK00004,10,9,3,0.333333
4,BOOK00005,10,9,4,0.444444



SOURCE EXPOSURE
Shape: (3, 6)

Columns:
 - source_group
 - catalog_books
 - catalog_pct
 - recommendation_count
 - recommendation_pct
 - exposure_difference_pct_points

Preview:


,source_group,catalog_books,catalog_pct,recommendation_count,recommendation_pct,exposure_difference_pct_points
0,Both,3,0.145138,22,0.113425,-0.031712
1,LeadershipNow only,1117,54.039671,8774,45.236131,-8.803540
2,Open Library only,947,45.815191,10600,54.650443,8.835252



RETRIEVAL DIAGNOSTIC
Shape: (6, 9)

Columns:
 - query
 - recommendations
 - unique_books
 - mean_similarity
 - max_similarity
 - unique_clusters
 - dominant_cluster
 - dominant_cluster_count
 - dominant_cluster_pct

Preview:


,query,recommendations,unique_books,mean_similarity,max_similarity,unique_clusters,dominant_cluster,dominant_cluster_count,dominant_cluster_pct
0,New Manager,10,10,0.239194,0.475328,5,Organizational Culture and Trust,5,50.0
1,Conflict Management,10,10,0.194382,0.314164,5,Organizational Culture and Trust,4,40.0
2,Strategy,10,10,0.281884,0.361138,4,Decision Making,6,60.0
3,Organizational Change,10,10,0.239024,0.364433,4,Change Management,7,70.0
4,Entrepreneurship,10,10,0.239672,0.318541,3,Entrepreneurial and Growth Mindset,7,70.0



RETRIEVAL ROBUSTNESS
Shape: (4, 6)

Columns:
 - test
 - query
 - recommendations
 - unique_books
 - all_positive_similarity
 - top_result

Preview:


,test,query,recommendations,unique_books,all_positive_similarity,top_result
0,Empty Query,NaN,0,0,NaN,NaN
1,Out of Vocabulary,qxzvplm jjjkkk zzzqqq,0,0,NaN,NaN
2,Focused Query,negotiation and conflict resolution,5,5,True,The Seven Tensions of Negotiation
3,Broad Query,leadership management,5,5,True,Leadership Development Studies


## 12. Recommendation Performance KPI Dataset

The recommendation-performance KPI table consolidates the principal evaluation measures for the final production recommender.

The measures are derived from the validated recommendation-evaluation artifacts rather than manually entered dashboard values.

The final recommendation system is evaluated using retrieval-oriented measures because no user-relevance labels or supervised target variable are available.

The KPI layer summarizes:

- catalog coverage;
- recommendation availability;
- recommendation similarity;
- topic diversity.

Source exposure and natural-language retrieval are retained as separate datasets because they describe different aspects of system behavior.

In [14]:
# ============================================================
# DASHBOARD 3 — RECOMMENDATION PERFORMANCE KPIs
# ============================================================

recommendation_pairs = pd.read_csv(
    PROCESSED_DIR
    / "recommendation_similarity_evaluation.csv"
)

CATALOG_SIZE = 2067

eligible_queries = (
    recommendation_diversity[
        "query_book_id"
    ].nunique()
)

unique_recommended = (
    recommendation_pairs[
        "recommended_book_id"
    ].nunique()
)


tableau_recommendation_kpis = pd.DataFrame([
    {
        "kpi": "Catalog Size",
        "value": CATALOG_SIZE,
        "unit": "books"
    },
    {
        "kpi": "Eligible Query Books",
        "value": eligible_queries,
        "unit": "books"
    },
    {
        "kpi": "Recommendation Pairs",
        "value": len(recommendation_pairs),
        "unit": "pairs"
    },
    {
        "kpi": "Unique Books Recommended",
        "value": unique_recommended,
        "unit": "books"
    },
    {
        "kpi": "Catalog Coverage",
        "value": (
            unique_recommended
            / CATALOG_SIZE
            * 100
        ),
        "unit": "%"
    },
    {
        "kpi": "Mean Similarity",
        "value": recommendation_pairs[
            "similarity_score"
        ].mean(),
        "unit": "cosine similarity"
    },
    {
        "kpi": "Median Similarity",
        "value": recommendation_pairs[
            "similarity_score"
        ].median(),
        "unit": "cosine similarity"
    },
    {
        "kpi": "Mean Rank-1 Similarity",
        "value": recommendation_pairs.loc[
            recommendation_pairs[
                "recommendation_rank"
            ] == 1,
            "similarity_score"
        ].mean(),
        "unit": "cosine similarity"
    },
    {
        "kpi": "Mean Rank-10 Similarity",
        "value": recommendation_pairs.loc[
            recommendation_pairs[
                "recommendation_rank"
            ] == 10,
            "similarity_score"
        ].mean(),
        "unit": "cosine similarity"
    },
    {
        "kpi": "Mean Unique Topic Clusters",
        "value": recommendation_diversity[
            "unique_topic_clusters"
        ].mean(),
        "unit": "clusters"
    },
    {
        "kpi": "Mean Cluster Diversity Ratio",
        "value": recommendation_diversity[
            "cluster_diversity_ratio"
        ].mean(),
        "unit": "ratio"
    }
])


print("RECOMMENDATION PERFORMANCE KPIs")
print("=" * 90)

display(
    tableau_recommendation_kpis
)

RECOMMENDATION PERFORMANCE KPIs


,kpi,value,unit
0,Catalog Size,2067.000000,books
1,Eligible Query Books,2040.000000,books
2,Recommendation Pairs,19396.000000,pairs
3,Unique Books Recommended,2016.000000,books
4,Catalog Coverage,97.532656,%
5,Mean Similarity,0.321981,cosine similarity
6,Median Similarity,0.301556,cosine similarity
7,Mean Rank-1 Similarity,0.513143,cosine similarity
8,Mean Rank-10 Similarity,0.233117,cosine similarity
9,Mean Unique Topic Clusters,3.840686,clusters


## 13. Tableau Recommendation Performance Tables

Four additional datasets are prepared for the performance dashboard.

### Similarity by Rank

Shows how recommendation similarity changes from Rank 1 through Rank 10.

### Topic Diversity

Retains one row per eligible query book so Tableau can visualize the distribution of topic diversity rather than only its average.

### Source Exposure

Compares each source's share of the catalog with its share of generated recommendations.

### Natural-Language Retrieval

Contains diagnostic and robustness results demonstrating how the retrieval system responds to different user intents and invalid inputs.

In [15]:
# ============================================================
# PREPARE DASHBOARD 3 TABLEAU TABLES
# ============================================================

# Already aggregated and validated
tableau_similarity_by_rank = (
    similarity_by_rank
    .copy()
    .sort_values("recommendation_rank")
    .reset_index(drop=True)
)


# One row per eligible query book
tableau_recommendation_diversity = (
    recommendation_diversity
    .copy()
)


# Add query-book metadata for Tableau filtering/tooltips
query_lookup = (
    tableau_catalog[
        [
            "book_id",
            "canonical_title",
            "authors",
            "source_group",
            "cluster_label_display"
        ]
    ]
    .rename(
        columns={
            "book_id": "query_book_id",
            "canonical_title": "query_title",
            "authors": "query_authors",
            "source_group": "query_source_group",
            "cluster_label_display": "query_topic"
        }
    )
)


tableau_recommendation_diversity = (
    tableau_recommendation_diversity
    .merge(
        query_lookup,
        on="query_book_id",
        how="left",
        validate="one_to_one"
    )
)


# Source exposure
tableau_source_exposure = (
    source_exposure.copy()
)


# Natural-language diagnostic
tableau_retrieval_diagnostic = (
    retrieval_diagnostic.copy()
)


# Robustness
tableau_retrieval_robustness = (
    retrieval_robustness.copy()
)


print("DASHBOARD 3 TABLES")
print("=" * 90)

print(
    "Similarity by rank:",
    tableau_similarity_by_rank.shape
)

print(
    "Recommendation diversity:",
    tableau_recommendation_diversity.shape
)

print(
    "Source exposure:",
    tableau_source_exposure.shape
)

print(
    "Retrieval diagnostic:",
    tableau_retrieval_diagnostic.shape
)

print(
    "Retrieval robustness:",
    tableau_retrieval_robustness.shape
)


print("\nDIVERSITY METADATA VALIDATION")
print("-" * 90)

print(
    "Unique query books:",
    tableau_recommendation_diversity[
        "query_book_id"
    ].nunique()
)

print(
    "Missing query titles:",
    tableau_recommendation_diversity[
        "query_title"
    ].isna().sum()
)

print(
    "Missing query sources:",
    tableau_recommendation_diversity[
        "query_source_group"
    ].isna().sum()
)

print(
    "Mean unique topic clusters:",
    tableau_recommendation_diversity[
        "unique_topic_clusters"
    ].mean()
)

print(
    "Mean diversity ratio:",
    tableau_recommendation_diversity[
        "cluster_diversity_ratio"
    ].mean()
)

DASHBOARD 3 TABLES
Similarity by rank: (10, 4)
Recommendation diversity: (2040, 9)
Source exposure: (3, 6)
Retrieval diagnostic: (6, 9)
Retrieval robustness: (4, 6)

DIVERSITY METADATA VALIDATION
------------------------------------------------------------------------------------------
Unique query books: 2040
Missing query titles: 0
Missing query sources: 0
Mean unique topic clusters: 3.840686274509804
Mean diversity ratio: 0.46730508870214754


## 14. Export Dashboard 3 — Recommendation System Performance

The validated recommendation-system evaluation datasets are exported for Tableau.

Six purpose-specific files are retained because they operate at different analytical grains:

1. recommendation KPIs — one row per performance indicator;
2. similarity by rank — one row per recommendation rank;
3. topic diversity — one row per eligible query book;
4. source exposure — one row per source group;
5. natural-language retrieval diagnostics — one row per diagnostic query;
6. retrieval robustness — one row per robustness test.

Keeping these datasets separate avoids duplicating aggregated values across incompatible levels of analysis.

In [16]:
# ============================================================
# EXPORT DASHBOARD 3 TABLEAU DATA
# ============================================================

RECOMMENDATION_KPI_PATH = (
    TABLEAU_DIR
    / "tableau_recommendation_kpis.csv"
)

SIMILARITY_RANK_PATH = (
    TABLEAU_DIR
    / "tableau_similarity_by_rank.csv"
)

DIVERSITY_PATH = (
    TABLEAU_DIR
    / "tableau_recommendation_diversity.csv"
)

SOURCE_EXPOSURE_PATH = (
    TABLEAU_DIR
    / "tableau_source_exposure.csv"
)

RETRIEVAL_DIAGNOSTIC_PATH = (
    TABLEAU_DIR
    / "tableau_retrieval_diagnostic.csv"
)

RETRIEVAL_ROBUSTNESS_PATH = (
    TABLEAU_DIR
    / "tableau_retrieval_robustness.csv"
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

tableau_recommendation_kpis.to_csv(
    RECOMMENDATION_KPI_PATH,
    index=False
)

tableau_similarity_by_rank.to_csv(
    SIMILARITY_RANK_PATH,
    index=False
)

tableau_recommendation_diversity.to_csv(
    DIVERSITY_PATH,
    index=False
)

tableau_source_exposure.to_csv(
    SOURCE_EXPOSURE_PATH,
    index=False
)

tableau_retrieval_diagnostic.to_csv(
    RETRIEVAL_DIAGNOSTIC_PATH,
    index=False
)

tableau_retrieval_robustness.to_csv(
    RETRIEVAL_ROBUSTNESS_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

kpi_reload = pd.read_csv(
    RECOMMENDATION_KPI_PATH
)

rank_reload = pd.read_csv(
    SIMILARITY_RANK_PATH
)

diversity_reload = pd.read_csv(
    DIVERSITY_PATH
)

exposure_reload = pd.read_csv(
    SOURCE_EXPOSURE_PATH
)

diagnostic_reload = pd.read_csv(
    RETRIEVAL_DIAGNOSTIC_PATH
)

robustness_reload = pd.read_csv(
    RETRIEVAL_ROBUSTNESS_PATH
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

dashboard3_checks = {
    "11 recommendation KPIs":
        len(kpi_reload) == 11,

    "10 recommendation ranks":
        len(rank_reload) == 10,

    "Ranks 1 through 10 present":
        set(rank_reload["recommendation_rank"]) == set(range(1, 11)),

    "2040 eligible query books":
        len(diversity_reload) == 2040,

    "2040 unique query books":
        diversity_reload["query_book_id"].nunique() == 2040,

    "3 source groups":
        len(exposure_reload) == 3,

    "Catalog source shares total approximately 100%":
        np.isclose(
            exposure_reload["catalog_pct"].sum(),
            100.0
        ),

    "Recommendation source shares total approximately 100%":
        np.isclose(
            exposure_reload["recommendation_pct"].sum(),
            100.0
        ),

    "6 diagnostic queries":
        len(diagnostic_reload) == 6,

    "4 robustness tests":
        len(robustness_reload) == 4,

    "Catalog coverage is 97.532656%":
        np.isclose(
            kpi_reload.loc[
                kpi_reload["kpi"] == "Catalog Coverage",
                "value"
            ].iloc[0],
            97.532656,
            atol=1e-6
        )
}


dashboard3_validation = pd.DataFrame(
    dashboard3_checks.items(),
    columns=[
        "validation_check",
        "passed"
    ]
)


print("DASHBOARD 3 TABLEAU EXPORT")
print("=" * 100)

print("Recommendation KPIs:", kpi_reload.shape)
print("Similarity by rank:", rank_reload.shape)
print("Topic diversity:", diversity_reload.shape)
print("Source exposure:", exposure_reload.shape)
print("Retrieval diagnostic:", diagnostic_reload.shape)
print("Retrieval robustness:", robustness_reload.shape)

print("\nVALIDATION")
print("-" * 100)

display(dashboard3_validation)

print(
    "\nAll Dashboard 3 checks passed:",
    bool(
        dashboard3_validation["passed"].all()
    )
)

DASHBOARD 3 TABLEAU EXPORT
Recommendation KPIs: (11, 3)
Similarity by rank: (10, 4)
Topic diversity: (2040, 9)
Source exposure: (3, 6)
Retrieval diagnostic: (6, 9)
Retrieval robustness: (4, 6)

VALIDATION
----------------------------------------------------------------------------------------------------


,validation_check,passed
0,11 recommendation KPIs,True
1,10 recommendation ranks,True
2,Ranks 1 through 10 present,True
3,2040 eligible query books,True
4,2040 unique query books,True
5,3 source groups,True
6,Catalog source shares total approximately 100%,True
7,Recommendation source shares total approximate...,True
8,6 diagnostic queries,True
9,4 robustness tests,True



All Dashboard 3 checks passed: True


## 15. Dashboard 4 — Model Evaluation and Final Architecture

The final Tableau dashboard summarizes the evaluation and role of each analytical component developed during the project.

The system contains multiple models and representations serving different purposes. Their evaluation metrics are therefore presented separately rather than combined into a single universal performance score.

The final architecture distinguishes between:

- **Production components** — used directly by the recommendation system;
- **Supporting components** — used for semantic representation or interpretability;
- **Experimental components** — evaluated but not selected for production;
- **Optional components** — designed to enhance explanation without determining recommendation ranking.

The final production recommendation engine remains:

**Enriched TF-IDF → Cosine Similarity**

The 200-dimensional LSA representation and 29-cluster K-Means solution provide supporting analytical structure.

The neural autoencoder remains experimental because reconstruction performance was strong while preservation of semantic neighborhood structure was substantially weaker.

The grounded LLM layer remains optional and operates in offline-grounding mode because no live LLM API inference was performed during the project.

In [17]:
# ============================================================
# DASHBOARD 4 — INSPECT MODEL EVALUATION INPUTS
# ============================================================

final_model_evaluation = pd.read_csv(
    PROCESSED_DIR
    / "final_model_evaluation_summary.csv"
)

svd_search = pd.read_csv(
    PROCESSED_DIR
    / "svd_component_search.csv"
)

autoencoder_evaluation = pd.read_csv(
    PROCESSED_DIR
    / "autoencoder_evaluation_summary.csv"
)

autoencoder_history = pd.read_csv(
    PROCESSED_DIR
    / "autoencoder_training_history.csv"
)


evaluation_inputs = {
    "Final Model Evaluation": final_model_evaluation,
    "SVD Component Search": svd_search,
    "Autoencoder Evaluation": autoencoder_evaluation,
    "Autoencoder Training History": autoencoder_history
}


for name, df in evaluation_inputs.items():

    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)

    print("Shape:", df.shape)

    print("\nColumns:")
    for column in df.columns:
        print(" -", column)

    print("\nPreview:")
    display(df.head(10))


FINAL MODEL EVALUATION
Shape: (6, 8)

Columns:
 - component
 - primary_function
 - key_metric
 - metric_value
 - metric_unit
 - supporting_evidence
 - main_limitation
 - final_role

Preview:


,component,primary_function,key_metric,metric_value,metric_unit,supporting_evidence,main_limitation,final_role
0,Enriched TF-IDF + Cosine,Production recommendation retrieval,Catalog coverage,97.532656,%,"2,016 unique books recommended; mean similarit...",Lexical sensitivity and heterogeneous metadata,Production
1,Enriched SVD / LSA (200D),Dense semantic representation,Top-10 neighbor preservation,57.966667,%,Similarity Spearman rho 0.516; 41.95% explaine...,Reduced representation approximates original T...,Supporting
2,K-Means (29 clusters),Topic discovery and semantic organization,Books clustered,1884.000000,books,29 interpretable topic clusters; median cluste...,Cluster cohesion varies across topics,Supporting
3,Autoencoder (200→32→200),Experimental nonlinear compression,Test MSE improvement,82.247979,%,Final test MSE 0.001188; semantic rho 0.107; T...,Weak preservation of original semantic geometry,Experimental
4,Natural-Language Retrieval,User-intent query interface,Successful diagnostic retrieval,6.000000,queries,10 unique recommendations per diagnostic; empt...,Broad and multi-concept queries can be less sp...,Production Interface
5,Grounded LLM Layer,Recommendation explanation,Grounding catalog coverage,2067.000000,books,Ranking-preserving prompt; anti-fabrication ru...,No live LLM inference evaluated,Optional Explanation



SVD COMPONENT SEARCH
Shape: (14, 3)

Columns:
 - representation
 - components
 - explained_variance_pct

Preview:


,representation,components,explained_variance_pct
0,Core,10,8.767354
1,Core,25,16.685701
2,Core,50,24.194685
3,Core,75,29.797957
4,Core,100,34.661219
5,Core,150,42.814084
6,Core,200,49.432955
7,Enriched,10,7.366644
8,Enriched,25,12.953342
9,Enriched,50,19.335067



AUTOENCODER EVALUATION
Shape: (11, 2)

Columns:
 - metric
 - value

Preview:


,metric,value
0,untrained_validation_mse,6.750306e-03
1,best_validation_mse,1.240390e-03
2,untrained_test_mse,6.692160e-03
3,final_test_mse,1.187994e-03
4,test_mse_improvement_pct,8.224798e+01
5,pairwise_similarity_spearman_rho,1.074673e-01
6,pairwise_similarity_spearman_p,6.745561e-120
7,mean_top10_shared_neighbors,4.274510e+00
8,median_top10_shared_neighbors,4.000000e+00
9,mean_top10_neighbor_preservation,4.274510e-01



AUTOENCODER TRAINING HISTORY
Shape: (50, 3)

Columns:
 - epoch
 - training_mse
 - validation_mse

Preview:


,epoch,training_mse,validation_mse
0,1,0.002669,0.002175
1,2,0.002059,0.002150
2,3,0.002025,0.002107
3,4,0.001953,0.002033
4,5,0.001848,0.001934
5,6,0.001747,0.001839
6,7,0.001663,0.001775
7,8,0.001598,0.001725
8,9,0.001543,0.001680
9,10,0.001495,0.001641


In [18]:
# ============================================================
# INSPECT FINAL MODEL ARCHITECTURE
# ============================================================

import json

ARCHITECTURE_PATH = (
    PROJECT_ROOT
    / "models"
    / "final_model_architecture.json"
)

with open(
    ARCHITECTURE_PATH,
    "r",
    encoding="utf-8"
) as file:
    final_architecture = json.load(file)


print("FINAL MODEL ARCHITECTURE")
print("=" * 100)

print(
    json.dumps(
        final_architecture,
        indent=2,
        ensure_ascii=False
    )
)

FINAL MODEL ARCHITECTURE
{
  "production_recommender": {
    "representation": "Enriched TF-IDF",
    "ranking_method": "Cosine Similarity",
    "catalog_size": 2067,
    "eligible_books": 2040,
    "catalog_coverage_pct": 97.532656
  },
  "semantic_representation": {
    "method": "Truncated SVD / LSA",
    "representation": "Enriched",
    "dimensions": 200,
    "similarity_spearman_rho": 0.515777,
    "top10_neighbor_preservation_pct": 57.966667
  },
  "topic_model": {
    "method": "K-Means",
    "clusters": 29,
    "books_clustered": 1884,
    "role": "Supplementary topic metadata"
  },
  "neural_network": {
    "method": "Autoencoder",
    "architecture": "200-128-64-32-64-128-200",
    "latent_dimensions": 32,
    "test_mse_improvement_pct": 82.247979,
    "similarity_spearman_rho": 0.1074673,
    "top10_neighbor_preservation_pct": 42.7451,
    "role": "Experimental"
  },
  "natural_language_retrieval": {
    "enabled": true,
    "representation": "Enriched TF-IDF",
    "ranking

## 17. Prepare Model Evaluation and Architecture Tables

The final dashboard uses separate datasets for model roles and model-specific evaluation.

The six-component evaluation summary provides the high-level architecture and final role of each analytical component.

Detailed SVD and autoencoder results are retained separately because their metrics have different meanings and should not be directly compared on a common performance scale.

The dashboard therefore emphasizes:

- final system architecture;
- production, supporting, experimental, and optional roles;
- dimensionality-reduction trade-offs;
- neural-network training behavior;
- semantic-preservation limitations.

No composite model score is calculated.

In [19]:
# ============================================================
# PREPARE DASHBOARD 4 TABLEAU TABLES
# ============================================================

# ------------------------------------------------------------
# 1. Final model architecture / evaluation summary
# ------------------------------------------------------------

tableau_model_architecture = (
    final_model_evaluation
    .copy()
)


# Add display order for Tableau
component_order = {
    "Enriched TF-IDF + Cosine": 1,
    "Enriched SVD / LSA (200D)": 2,
    "K-Means (29 clusters)": 3,
    "Autoencoder (200→32→200)": 4,
    "Natural-Language Retrieval": 5,
    "Grounded LLM Layer": 6
}

tableau_model_architecture["display_order"] = (
    tableau_model_architecture["component"]
    .map(component_order)
)


# ------------------------------------------------------------
# 2. SVD component evaluation
# ------------------------------------------------------------

tableau_svd_evaluation = (
    svd_search
    .copy()
    .sort_values(
        ["representation", "components"]
    )
    .reset_index(drop=True)
)

tableau_svd_evaluation["selected_model"] = (
    (tableau_svd_evaluation["representation"] == "Enriched")
    &
    (tableau_svd_evaluation["components"] == 200)
)


# ------------------------------------------------------------
# 3. Autoencoder evaluation
# ------------------------------------------------------------

tableau_autoencoder_evaluation = (
    autoencoder_evaluation.copy()
)


# ------------------------------------------------------------
# 4. Autoencoder training history
# ------------------------------------------------------------

tableau_autoencoder_history = (
    autoencoder_history
    .copy()
    .sort_values("epoch")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("DASHBOARD 4 TABLES")
print("=" * 100)

print(
    "Model architecture:",
    tableau_model_architecture.shape
)

print(
    "SVD evaluation:",
    tableau_svd_evaluation.shape
)

print(
    "Autoencoder evaluation:",
    tableau_autoencoder_evaluation.shape
)

print(
    "Autoencoder history:",
    tableau_autoencoder_history.shape
)


print("\nFINAL MODEL ROLES")
print("-" * 100)

display(
    tableau_model_architecture[
        [
            "display_order",
            "component",
            "primary_function",
            "key_metric",
            "metric_value",
            "metric_unit",
            "final_role"
        ]
    ]
    .sort_values("display_order")
    .reset_index(drop=True)
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Six architecture components:",
    len(tableau_model_architecture) == 6
)

print(
    "All components have roles:",
    tableau_model_architecture[
        "final_role"
    ].notna().all()
)

print(
    "Display order complete:",
    tableau_model_architecture[
        "display_order"
    ].notna().all()
)

print(
    "SVD candidate rows:",
    len(tableau_svd_evaluation)
)

print(
    "Selected SVD configuration count:",
    tableau_svd_evaluation[
        "selected_model"
    ].sum()
)

print(
    "Autoencoder evaluation metrics:",
    len(tableau_autoencoder_evaluation)
)

print(
    "Autoencoder epochs:",
    len(tableau_autoencoder_history)
)

DASHBOARD 4 TABLES
Model architecture: (6, 9)
SVD evaluation: (14, 4)
Autoencoder evaluation: (11, 2)
Autoencoder history: (50, 3)

FINAL MODEL ROLES
----------------------------------------------------------------------------------------------------


,display_order,component,primary_function,key_metric,metric_value,metric_unit,final_role
0,1,Enriched TF-IDF + Cosine,Production recommendation retrieval,Catalog coverage,97.532656,%,Production
1,2,Enriched SVD / LSA (200D),Dense semantic representation,Top-10 neighbor preservation,57.966667,%,Supporting
2,3,K-Means (29 clusters),Topic discovery and semantic organization,Books clustered,1884.000000,books,Supporting
3,4,Autoencoder (200→32→200),Experimental nonlinear compression,Test MSE improvement,82.247979,%,Experimental
4,5,Natural-Language Retrieval,User-intent query interface,Successful diagnostic retrieval,6.000000,queries,Production Interface
5,6,Grounded LLM Layer,Recommendation explanation,Grounding catalog coverage,2067.000000,books,Optional Explanation



VALIDATION
----------------------------------------------------------------------------------------------------
Six architecture components: True
All components have roles: True
Display order complete: True
SVD candidate rows: 14
Selected SVD configuration count: 1
Autoencoder evaluation metrics: 11
Autoencoder epochs: 50


In [20]:
# ============================================================
# EXPORT DASHBOARD 4 TABLEAU DATA
# ============================================================

MODEL_ARCHITECTURE_PATH = (
    TABLEAU_DIR
    / "tableau_model_architecture.csv"
)

SVD_EVALUATION_PATH = (
    TABLEAU_DIR
    / "tableau_svd_evaluation.csv"
)

AUTOENCODER_EVALUATION_PATH = (
    TABLEAU_DIR
    / "tableau_autoencoder_evaluation.csv"
)

AUTOENCODER_HISTORY_PATH = (
    TABLEAU_DIR
    / "tableau_autoencoder_training_history.csv"
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

tableau_model_architecture.to_csv(
    MODEL_ARCHITECTURE_PATH,
    index=False
)

tableau_svd_evaluation.to_csv(
    SVD_EVALUATION_PATH,
    index=False
)

tableau_autoencoder_evaluation.to_csv(
    AUTOENCODER_EVALUATION_PATH,
    index=False
)

tableau_autoencoder_history.to_csv(
    AUTOENCODER_HISTORY_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

architecture_reload = pd.read_csv(
    MODEL_ARCHITECTURE_PATH
)

svd_reload = pd.read_csv(
    SVD_EVALUATION_PATH
)

autoencoder_eval_reload = pd.read_csv(
    AUTOENCODER_EVALUATION_PATH
)

autoencoder_history_reload = pd.read_csv(
    AUTOENCODER_HISTORY_PATH
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

dashboard4_checks = {
    "6 model components":
        len(architecture_reload) == 6,

    "All model roles present":
        architecture_reload[
            "final_role"
        ].notna().all(),

    "Production recommender present":
        (
            architecture_reload["component"]
            == "Enriched TF-IDF + Cosine"
        ).any(),

    "Autoencoder remains experimental":
        architecture_reload.loc[
            architecture_reload["component"]
            == "Autoencoder (200→32→200)",
            "final_role"
        ].iloc[0] == "Experimental",

    "14 SVD candidate rows":
        len(svd_reload) == 14,

    "One selected SVD configuration":
        svd_reload[
            "selected_model"
        ].sum() == 1,

    "Selected SVD is Enriched 200D":
        (
            (
                svd_reload["representation"]
                == "Enriched"
            )
            &
            (
                svd_reload["components"]
                == 200
            )
            &
            (
                svd_reload["selected_model"]
                == True
            )
        ).sum() == 1,

    "11 autoencoder evaluation metrics":
        len(autoencoder_eval_reload) == 11,

    "50 autoencoder training epochs":
        len(autoencoder_history_reload) == 50
}


dashboard4_validation = pd.DataFrame(
    dashboard4_checks.items(),
    columns=[
        "validation_check",
        "passed"
    ]
)


print("DASHBOARD 4 TABLEAU EXPORT")
print("=" * 100)

print(
    "Model architecture:",
    architecture_reload.shape
)

print(
    "SVD evaluation:",
    svd_reload.shape
)

print(
    "Autoencoder evaluation:",
    autoencoder_eval_reload.shape
)

print(
    "Autoencoder history:",
    autoencoder_history_reload.shape
)

print("\nVALIDATION")
print("-" * 100)

display(
    dashboard4_validation
)

print(
    "\nAll Dashboard 4 checks passed:",
    bool(
        dashboard4_validation["passed"].all()
    )
)

DASHBOARD 4 TABLEAU EXPORT
Model architecture: (6, 9)
SVD evaluation: (14, 4)
Autoencoder evaluation: (11, 2)
Autoencoder history: (50, 3)

VALIDATION
----------------------------------------------------------------------------------------------------


,validation_check,passed
0,6 model components,True
1,All model roles present,True
2,Production recommender present,True
3,Autoencoder remains experimental,True
4,14 SVD candidate rows,True
5,One selected SVD configuration,True
6,Selected SVD is Enriched 200D,True
7,11 autoencoder evaluation metrics,True
8,50 autoencoder training epochs,True



All Dashboard 4 checks passed: True


## 19. Final Tableau Export Audit

The final stage of Tableau data preparation verifies all exported visualization datasets.

The audit confirms that:

- all expected Tableau files exist;
- each dataset contains records;
- the four dashboard data groups are complete;
- catalog and clustering counts remain consistent with the validated analytical pipeline;
- recommendation-system evaluation outputs retain their expected dimensions;
- final model-evaluation datasets preserve the selected production and supporting architecture.

This audit does not retrain, modify, or reevaluate any model. It verifies only the integrity of the visualization-ready exports.

In [21]:
# ============================================================
# FINAL TABLEAU EXPORT AUDIT
# ============================================================

expected_tableau_files = {

    # Dashboard 1 — Catalog Overview
    "Dashboard 1": [
        "tableau_catalog_overview.csv",
        "tableau_catalog_kpis.csv",
        "tableau_metadata_coverage.csv"
    ],

    # Dashboard 2 — Topic Landscape
    "Dashboard 2": [
        "tableau_topic_clusters.csv",
        "tableau_representative_books.csv"
    ],

    # Dashboard 3 — Recommendation Performance
    "Dashboard 3": [
        "tableau_recommendation_kpis.csv",
        "tableau_similarity_by_rank.csv",
        "tableau_recommendation_diversity.csv",
        "tableau_source_exposure.csv",
        "tableau_retrieval_diagnostic.csv",
        "tableau_retrieval_robustness.csv"
    ],

    # Dashboard 4 — Model Evaluation
    "Dashboard 4": [
        "tableau_model_architecture.csv",
        "tableau_svd_evaluation.csv",
        "tableau_autoencoder_evaluation.csv",
        "tableau_autoencoder_training_history.csv"
    ]
}


audit_rows = []

for dashboard, filenames in expected_tableau_files.items():

    for filename in filenames:

        path = TABLEAU_DIR / filename

        exists = path.exists()

        rows = None
        columns = None
        non_empty = False

        if exists:
            df = pd.read_csv(path)

            rows = len(df)
            columns = len(df.columns)
            non_empty = rows > 0

        audit_rows.append({
            "dashboard": dashboard,
            "file": filename,
            "exists": exists,
            "rows": rows,
            "columns": columns,
            "non_empty": non_empty
        })


tableau_export_audit = pd.DataFrame(audit_rows)


print("FINAL TABLEAU EXPORT AUDIT")
print("=" * 110)

display(tableau_export_audit)


print("\nSUMMARY")
print("-" * 110)

print(
    "Expected files:",
    len(tableau_export_audit)
)

print(
    "Files found:",
    int(tableau_export_audit["exists"].sum())
)

print(
    "Non-empty files:",
    int(tableau_export_audit["non_empty"].sum())
)

print(
    "Missing files:",
    int((~tableau_export_audit["exists"]).sum())
)


print("\nFILES BY DASHBOARD")
print("-" * 110)

display(
    tableau_export_audit
    .groupby("dashboard")
    .agg(
        files=("file", "count"),
        files_found=("exists", "sum"),
        non_empty=("non_empty", "sum")
    )
    .reset_index()
)

FINAL TABLEAU EXPORT AUDIT


,dashboard,file,exists,rows,columns,non_empty
0,Dashboard 1,tableau_catalog_overview.csv,True,2067,38,True
1,Dashboard 1,tableau_catalog_kpis.csv,True,6,3,True
2,Dashboard 1,tableau_metadata_coverage.csv,True,10,4,True
3,Dashboard 2,tableau_topic_clusters.csv,True,29,17,True
4,Dashboard 2,tableau_representative_books.csv,True,145,14,True
5,Dashboard 3,tableau_recommendation_kpis.csv,True,11,3,True
6,Dashboard 3,tableau_similarity_by_rank.csv,True,10,4,True
7,Dashboard 3,tableau_recommendation_diversity.csv,True,2040,9,True
8,Dashboard 3,tableau_source_exposure.csv,True,3,6,True
9,Dashboard 3,tableau_retrieval_diagnostic.csv,True,6,9,True



SUMMARY
--------------------------------------------------------------------------------------------------------------
Expected files: 15
Files found: 15
Non-empty files: 15
Missing files: 0

FILES BY DASHBOARD
--------------------------------------------------------------------------------------------------------------


,dashboard,files,files_found,non_empty
0,Dashboard 1,3,3,3
1,Dashboard 2,2,2,2
2,Dashboard 3,6,6,6
3,Dashboard 4,4,4,4


In [22]:
# ============================================================
# FINAL NOTEBOOK 17 INTEGRITY CHECKS
# ============================================================

final_checks = {

    # Dashboard 1
    "Catalog contains 2067 books":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_catalog_overview.csv"
            )
        ) == 2067,

    "Catalog book IDs are unique":
        pd.read_csv(
            TABLEAU_DIR / "tableau_catalog_overview.csv"
        )["book_id"].nunique() == 2067,

    # Dashboard 2
    "Topic landscape contains 29 clusters":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_topic_clusters.csv"
            )
        ) == 29,

    "Topic clusters represent 1884 books":
        pd.read_csv(
            TABLEAU_DIR / "tableau_topic_clusters.csv"
        )["cluster_size"].sum() == 1884,

    "Representative table contains 145 books":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_representative_books.csv"
            )
        ) == 145,

    # Dashboard 3
    "Recommendation evaluation contains 2040 queries":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_recommendation_diversity.csv"
            )
        ) == 2040,

    "Similarity table contains ranks 1-10":
        set(
            pd.read_csv(
                TABLEAU_DIR / "tableau_similarity_by_rank.csv"
            )["recommendation_rank"]
        ) == set(range(1, 11)),

    "Source exposure contains 3 groups":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_source_exposure.csv"
            )
        ) == 3,

    # Dashboard 4
    "Architecture contains 6 components":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_model_architecture.csv"
            )
        ) == 6,

    "SVD evaluation contains 14 candidates":
        len(
            pd.read_csv(
                TABLEAU_DIR / "tableau_svd_evaluation.csv"
            )
        ) == 14,

    "Autoencoder history contains 50 epochs":
        len(
            pd.read_csv(
                TABLEAU_DIR
                / "tableau_autoencoder_training_history.csv"
            )
        ) == 50,

    # Entire Tableau layer
    "All 15 expected Tableau files exist":
        tableau_export_audit["exists"].all(),

    "All Tableau files are non-empty":
        tableau_export_audit["non_empty"].all()
}


final_tableau_validation = pd.DataFrame(
    final_checks.items(),
    columns=[
        "validation_check",
        "passed"
    ]
)


print("NOTEBOOK 17 — FINAL INTEGRITY VALIDATION")
print("=" * 110)

display(final_tableau_validation)

print(
    "\nChecks passed:",
    int(final_tableau_validation["passed"].sum()),
    "/",
    len(final_tableau_validation)
)

print(
    "Notebook 17 Tableau data preparation complete:",
    bool(
        final_tableau_validation["passed"].all()
    )
)

NOTEBOOK 17 — FINAL INTEGRITY VALIDATION


,validation_check,passed
0,Catalog contains 2067 books,True
1,Catalog book IDs are unique,True
2,Topic landscape contains 29 clusters,True
3,Topic clusters represent 1884 books,True
4,Representative table contains 145 books,True
5,Recommendation evaluation contains 2040 queries,True
6,Similarity table contains ranks 1-10,True
7,Source exposure contains 3 groups,True
8,Architecture contains 6 components,True
9,SVD evaluation contains 14 candidates,True



Checks passed: 13 / 13
Notebook 17 Tableau data preparation complete: True


## 20. Final Summary

Notebook 17 prepared and validated the visualization-ready datasets for the Tableau stage of the Leadership and Management Book Recommendation System.

No machine-learning models were trained, modified, or retuned in this notebook. The notebook transformed previously validated analytical outputs into purpose-specific Tableau datasets while preserving the results of the earlier NLP, clustering, recommendation, neural-network, and model-evaluation stages.

### Dashboard 1 — Catalog Overview

The catalog dashboard represents the complete integrated collection of **2,067 canonical books**.

The exported datasets support analysis of:

- source composition;
- publication information;
- metadata availability;
- reader engagement;
- content depth;
- topic-cluster availability.

Metadata coverage also documents important limitations in the catalog, including relatively sparse descriptions, ratings, and reader-engagement information.

### Dashboard 2 — Leadership Topic Landscape

The topic-landscape dashboard represents the final unsupervised clustering solution:

- **1,884 clustered books**
- **29 topic clusters**
- **145 representative books**
- **5 representative books per cluster**

Cluster size, cohesion, source composition, characteristic terms, and representative titles are retained for visualization.

The topic clusters are interpreted as data-driven semantic groupings rather than definitive classifications of leadership and management literature.

### Dashboard 3 — Recommendation System Performance

The recommendation dashboard evaluates the final production recommendation system:

**Enriched TF-IDF + Cosine Similarity**

Key validated results include:

- **2,040 eligible query books**
- **19,396 recommendation pairs**
- **2,016 unique books recommended**
- **97.53% catalog coverage**
- **0.322 mean cosine similarity**
- **0.513 mean Rank-1 similarity**
- **0.233 mean Rank-10 similarity**
- **3.84 mean unique topic clusters per recommendation list**
- **0.467 mean cluster-diversity ratio**

The dashboard also presents source exposure and natural-language retrieval diagnostics.

Differences in recommendation exposure between Open Library and LeadershipNow are reported as source-associated behavior rather than automatically interpreted as algorithmic bias because metadata availability and richness differ substantially between the sources.

### Dashboard 4 — Model Evaluation and Final Architecture

The final dashboard summarizes the roles and evaluation of the project's analytical components.

The final architecture is:

- **Enriched TF-IDF + Cosine Similarity** — Production recommender
- **Enriched SVD / LSA (200 dimensions)** — Supporting semantic representation
- **K-Means (29 clusters)** — Supporting topic organization
- **Autoencoder (200 → 32 → 200)** — Experimental neural representation
- **Natural-Language Retrieval** — Production query interface
- **Grounded LLM Layer** — Optional explanation layer

The autoencoder achieved substantial reconstruction improvement but weaker semantic-geometry preservation and therefore was not selected to replace the production TF-IDF recommender.

The grounded LLM architecture remains in offline-grounding mode because live LLM API inference was not performed during the project.

### Tableau Export Validation

A total of **15 visualization-ready CSV files** were exported across the four dashboard groups.

Final validation confirmed:

- **15 / 15 expected files exist**
- **15 / 15 files are non-empty**
- **13 / 13 final integrity checks passed**

The Tableau data layer is therefore complete and ready for dashboard development.

---

**Notebook 17 Status: COMPLETE**

**Next Stage: Tableau Dashboard Development**